# Experiment 3: Volcanic Eruption

### Objective:
- Simulate the injection of volcanic ash and analyze their dispersion.
- Understand the role of altitude, wind, and turbulence in determining the impact of volcanic eruptions.
    
### Simulation Design:
- Eyjafjallajökull eruption (First explosive phase) : 14-18 April 2010
- Grid horizontal resolution - 40 km
- Inject ash as point source emission at different altitudes (4 km vs. 9 km).
- Run the model for 4 days to track transport.
    
### Analysis:
- Compare dispersion patterns of ash 
- Identify regions impacted by the volcanic plume.
- Discuss implications for aviation
- Discuss the limitations of our simulation: take a look at figure 2 a and c in https://www.nature.com/articles/srep00572. Which parts of the emission do we correctly capture, and which parts do we simplify a lot?

We will simulate the first explosive phase of the 2010 Eyjafjallajökull eruption with the following steps:

## 1. Prepare the XML Files for the emission
The xml files needed for this exercise are in the Exp03 folder in your home directory:
* Ex03_pntSrc.xml     : for point source emission
* Ex03_modes.xml      : for description of the (background) modes
* Ex03_aerotracer.xml : for description of the treated aerosol tracers

### Eruption source parameters to include in pntSrc.xml
- Location of eruption: 63.633°N & 19.633°W
- Emission duration: 14 April 2010, 01:30 UTC - 18 April, 2010 00:00 UTC 
- Ash injection heights: 4000 m & 9000 m **(in 2 different experiments)**

The emitted ash distribution consists of 3 modes with the following median diameters and standard deviations:

| | Insoluble Accumulation | Insoluble Coarse | Insoluble Giant |
|-----|-----|-----|-----|
| median diameter [m] (dg3_emiss)| 0.8E-6 | 2.98E-6 | 11.35E-6 |
| standard deviation (sigma_emiss) | 1.4 | 1.4 | 1.4 |
| emission strength [kg/s] (source_strength)| 7500 | 7500 | 7500 |

## 2. Link the input data

- ICON Grid: R02B06, Horizontal resolution - 40 km

In [ ]:
# base directory for ICON sources and binary:
export ICONDIR=/Path/to/ICON-model

# directory with initial conditions, external parameters and grid file
export INDIR=/Path/to/Input-data

# directory with xml files
export XMLDIR=/Path/to/xmlfiles

# absolute path to directory with plenty of space:
export OUTDIR=/Path/to/output-directoy

In [ ]:
# create output directory if not existing yet:
if [ ! -d $OUTDIR ]; then
    mkdir -p $OUTDIR
fi
cd $OUTDIR

# copy icon binary to OUTDIR
cp ${ICONDIR}/icon icon.exe

In [ ]:
# grid files
ln -sf $INDIR/icon_grid_0024_R02B06_G.nc iconR2B06_DOM01.nc
ln -sf $INDIR/icon_grid_0024_R02B06_G-grfinfo.nc iconR2B06_DOM01-grfinfo.nc
ln -sf $INDIR/icon_extpar_0024_R02B06_G_tiles.nc extpar_iconR2B06_DOM01.nc

# initial  data
ln -sf $INDIR/IFS_R2B06.nc IFS_R2B06.nc

# Link files needed for physics
ln -sf ${ICONDIR}/ECHAM6_CldOptProps.nc ECHAM6_CldOptProps.nc
ln -sf ${ICONDIR}/rrtmg_lw.nc rrtmg_lw.nc

## 3. Runscript

In [ ]:
# ----------------------------------------------------------------------------
# create ICON master namelist
# ----------------------------------------------------------------------------

cat > icon_master.namelist << EOF
! master_nml: ----------------------------------------------------------------
&master_nml
 lRestart                     =                    .false.        ! .TRUE.=current experiment is resumed
/
! master_time_control_nml: settings for start and duration times--------------
&master_time_control_nml
experimentStartDate           =     "2010-04-14T00:00:00Z"
experimentStopDate            =     "2010-04-18T00:00:00Z"
forecastLeadTime              =                      "P4D"
/
! master_model_nml: repeated for each model ----------------------------------
&master_model_nml
  model_type		           =	                    1         ! identifies which component to run (atmosphere,ocean,...)
  model_name		           =	                "ATMO"        ! character string for naming this component.
  model_namelist_filename      =            "NAMELIST_NWP"        ! file name containing the model namelists
  model_min_rank	           =	                    1         ! start MPI rank for this model
  model_max_rank	           =	                65536         ! end MPI rank for this model
  model_inc_rank	           =	                    1         ! stride of MPI ranks
/
EOF

**Remember to change which XML File is linked in cart_pntSrc_xml. Also change your output_filename from 'art-eyja-4000' to 'art-eyja-9000' for your second run! Otherwise the files will be overwritten and you'll need to run everything again**

In [ ]:
# ----------------------------------------------------------------------
# model namelists
# ----------------------------------------------------------------------

cat > NAMELIST_NWP << EOF
! parallel_nml: MPI parallelization -------------------------------------------
&parallel_nml
 nproma                      =                          8         ! loop chunk length
 p_test_run                  =                     .FALSE.        ! .TRUE. means verification run for MPI parallelization
 l_test_openmp               =                     .false.
 l_log_checks                =                     .false.
 num_io_procs                =                          1         ! number of I/O processors
 iorder_sendrecv             =                          3         ! sequence of MPI send/receive calls
/
! grid_nml: horizontal grid --------------------------------------------------
&grid_nml
 dynamics_grid_filename      =        "iconR2B06_DOM01.nc"        ! array of the grid filenames for the dycore
 dynamics_parent_grid_id     =                        0,1         ! array of the indexes of the parent grid filenames
 lfeedback                   =             .false.,.false.        ! specifies if feedback to parent grid is performed
 ifeedback_type              =                          2         ! feedback type (incremental/relaxation-based)
 lredgrid_phys               =                     .false.        ! .true.=radiation is calculated on a reduced grid
 start_time                  =                    0.,5400.        ! Time when a nested domain starts to be active [s]
/
! initicon_nml: specify read-in of initial state ------------------------------
&initicon_nml
 init_mode                   =                          2         ! operation mode 2: IFS 
 ifs2icon_filename           =              "IFS_R2B06.nc"        ! initial data filename
 lread_ana                   =                     .FALSE.        ! no analysis data will be read
 ltile_coldstart             =                      .TRUE.        ! coldstart for surface tiles
 ltile_init                  =                     .FALSE.        ! set it to .TRUE. if FG data originate from run without tiles
/
! run_nml: general switches ---------------------------------------------------
&run_nml
 num_lev                     =                      90,60         ! number of full levels (atm.) for each domain
 lvert_nest                  =                      .true.        ! use vertical nesting if a nest is active
 modelTimeStep               =                      "PT6M"
 ldynamics                   =                      .TRUE.        ! compute adiabatic dynamic tendencies
 ltransport                  =                      .TRUE.        ! compute large-scale tracer transport
 iforcing                    =                          3         ! forcing of dynamics and transport by parameterized processes
 ltestcase                   =                     .FALSE.        ! real case run
 msg_level                   =                          7         ! detailed report during integration
 ltimer                      =                      .TRUE.        ! timer for monitoring the runtime of specific routines
 timers_level                =                          1         ! can be increased up to 10 for detailed timer output
 output                      =                        "nml"       ! main switch for enabling/disabling components of the model output
 lart                        =                      .TRUE.
 check_uuid_gracefully       =                      .TRUE.        ! give only warnings for non-matching uuids
/
! nwp_phy_nml: switches for the physics schemes ------------------------------
&nwp_phy_nml
 inwp_gscp                   =                          1         ! cloud microphysics and precipitation
 inwp_convection             =                        1,1         ! convection
 inwp_radiation              =                          4         ! radiation
 inwp_cldcover               =                          1         ! cloud cover scheme for radiation
 inwp_turb                   =                          1         ! vertical diffusion and transfer
 inwp_satad                  =                          1         ! saturation adjustment
 inwp_sso                    =                          1         ! subgrid scale orographic drag
 inwp_gwd                    =                          1         ! non-orographic gravity wave drag
 inwp_surface                =                          1         ! surface scheme
 icapdcycl                   =                          3         ! apply CAPE modification to improve diurnalcycle over tropical land
 latm_above_top              =              .FALSE.,.TRUE.        ! take into account atmosphere above model top for radiation computation
 efdt_min_raylfric           =                       7200.        ! minimum e-folding time of Rayleigh friction
 itype_z0                    =                          2         ! type of roughness length data
 icpl_aero_conv              =                          0         ! coupling between autoconversion and Tegen aerosol climatology
 icpl_aero_gscp              =                          0         ! coupling between autoconversion and Tegen aerosol climatology
! resolution-dependent settings - please choose the appropriate one
 dt_rad                      =                        2160.       ! time step for radiation in s
 dt_conv                     =                         720.       ! time step for convection in s (domain specific)
 dt_sso                      =                        1440.       ! time step for SSO parameterization
 dt_gwd                      =                        1440.       ! time step for gravity wave drag parameterization
/
! nwp_tuning_nml: additional tuning parameters ----------------------------------
&nwp_tuning_nml
 tune_zceff_min              =                       0.05
 itune_albedo                =                          1         ! reduced albedo (w.r.t. MODIS data) over Sahara
/
! turbdiff_nml: turbulent diffusion -------------------------------------------
&turbdiff_nml
 a_hshr                      =                          2.0       ! length scale factor for separated horizontal shear mode
 frcsmot                     =                          0.2       ! these 2 switches together apply vertical smoothing of the TKE source terms
 icldm_turb                  =                            2       ! mode of cloud water representation in turbulence
 imode_frcsmot               =                            2       ! in the tropics (only), which reduces the moist bias in the tropical lower troposphere
 imode_tkesso                =                            2
 itype_sher                  =                            2       ! type of shear forcing used in turbulence
 ltkeshs                     =                        .TRUE.      ! include correction term for coarse grids in hor. shear production term
 ltkesso                     =                        .TRUE.      ! consider TKE-production by sub-grid SSO wakes
 pat_len                     =                        750.0       ! effective length scale of thermal surface patterns
 q_crit                      =                          2.0
 rat_sea                     =                          0.8
 tkhmin                      =                          0.5       ! scaling factor for minimum vertical diffusion coefficient
 tkmmin                      =                          0.75      ! scaling factor for minimum vertical diffusion coefficient
 tur_len                     =                          300.
 rlam_heat                   =                         10.0 
 alpha1                      =                         0.125
/
! lnd_nml: land scheme switches -----------------------------------------------
&lnd_nml
 ntiles                       =                           3       !!! 1 for assimilation cycle and forecast
 nlev_snow                    =                           1       !!! 1 for assimilation cycle and forecast
 lmulti_snow                  =                     .false.       !!! .false. for assimilation cycle and forecast
 itype_heatcond               =                           2       ! type of soil heat conductivity
 idiag_snowfrac               =                          20       ! type of snow-fraction diagnosis
 lsnowtile                    =                       .true.      ! .TRUE.=consider snow-covered and snow-free separately
 lseaice                      =                       .true.      ! .TRUE. for use of sea-ice model
 llake                        =                       .true.      ! .TRUE. for use of lake model
 frlake_thrhld                =                         0.05
 itype_lndtbl                 =                            3      ! minimizes moist/cold bias in lower tropical troposphere
 itype_root                   =                            2
/
! radiation_nml: radiation scheme ---------------------------------------------
&radiation_nml
 irad_o3                     =                          7         ! ozone climatology
 irad_aero                   =                          6         ! aerosols
 albedo_type                 =                          2         ! type of surface albedo
 vmr_co2                     =                    390.e-06
 vmr_ch4                     =                   1800.e-09
 vmr_n2o                     =                   322.0e-09
 vmr_o2                      =                     0.20946
 vmr_cfc11                   =                    240.e-12
 vmr_cfc12                   =                    532.e-12
 ecrad_data_path             =  '${ICONDIR}/externals/ecrad/data'
/
! nonhydrostatic_nml: nonhydrostatic model -----------------------------------
&nonhydrostatic_nml
 iadv_rhotheta               =                          2         ! advection method for rho and rhotheta
 ivctype                     =                          2         ! type of vertical coordinate
 itime_scheme                =                          4         ! time integration scheme
 exner_expol                 =                          0.333     ! temporal extrapolation of Exner function
 vwind_offctr                =                          0.2       ! off-centering in vertical wind solver
 damp_height                 =                      50000.0       ! height at which Rayleigh damping of vertical wind starts
 rayleigh_coeff              =                          0.10      ! Rayleigh damping coefficient
 divdamp_order               =                         24         ! order of divergence damping 
 divdamp_type                =                         32         ! type of divergence damping
 divdamp_fac                 =                          0.004     ! scaling factor for divergence damping
 igradp_method               =                          3         ! discretization of horizontal pressure gradient
 l_zdiffu_t                  =                      .TRUE.        ! specifies computation of Smagorinsky temperature diffusion
 thslp_zdiffu                =                          0.02      ! slope threshold (temperature diffusion)
 thhgtd_zdiffu               =                        125.0       ! threshold of height difference (temperature diffusion)
 htop_moist_proc             =                      22500.0       ! max. height for moist physics
 hbot_qvsubstep              =                      22500.0       ! height above which QV is advected with substepping scheme
/
! sleve_nml: vertical level specification -------------------------------------
&sleve_nml
 min_lay_thckn               =                         20.0       ! layer thickness of lowermost layer
 max_lay_thckn               =                        400.        ! maximum layer thickness below htop_thcknlimit
 htop_thcknlimit             =                      14000.
 top_height                  =                      75000.0       ! height of model top
 stretch_fac                 =                          0.9       ! stretching factor to vary distribution of model levels
 decay_scale_1               =                       4000.0       ! decay scale of large-scale topography component
 decay_scale_2               =                       2500.0       ! decay scale of small-scale topography component
 decay_exp                   =                          1.2       ! exponent of decay function
 flat_height                 =                      16000.0       ! height above which the coordinate surfaces are flat
/
! dynamics_nml: dynamical core -----------------------------------------------
&dynamics_nml
 iequations                  =                          3         ! type of equations and prognostic variables
 divavg_cntrwgt              =                          0.50      ! weight of central cell for divergence averaging
 lcoriolis                   =                      .TRUE.        ! Coriolis force
/
! transport_nml: tracer transport ---------------------------------------------
&transport_nml
 ctracer_list                =                      '12345'
 ivadv_tracer                =        3,3,3,3,3,3,3,3,3,3,3
 itype_hlimit                =        3,4,4,4,4,3,3,3,3,3,3
 ihadv_tracer                = 52,2,2,2,2,22,22,22,22,22,22
 iadv_tke                    =                            0
/
! diffusion_nml: horizontal (numerical) diffusion ----------------------------
&diffusion_nml
 hdiff_order                 =                          5         ! order of nabla operator for diffusion
 itype_vn_diffu              =                          1         ! reconstruction method used for Smagorinsky diffusion
 itype_t_diffu               =                          2         ! discretization of temperature diffusion
 hdiff_efdt_ratio            =                         24.0       ! ratio of e-folding time to time step 
 hdiff_smag_fac              =                          0.025     ! scaling factor for Smagorinsky diffusion
 lhdiff_vn                   =                      .TRUE.        ! diffusion on the horizontal wind field
 lhdiff_temp                 =                      .TRUE.        ! diffusion on the temperature field
/
! interpol_nml: settings for internal interpolation methods ------------------
&interpol_nml
 nudge_zone_width            =                          8         ! width of lateral boundary nudging zone
 lsq_high_ord                =                          3
 l_intp_c2l                  =                      .true.
 l_mono_c2l                  =                      .true.
 support_baryctr_intp        =                      .true.        ! barycentric interpolation support for output
/
&gribout_nml
 generatingCenter            = 78
 generatingSubcenter         = 255
/
! extpar_nml: external data --------------------------------------------------
&extpar_nml
 itopo                       =                          1         ! topography (0:analytical)
 n_iter_smooth_topo          =                          1         ! iterations of topography smoother
 heightdiff_threshold        =                       3000.        ! height difference between neighb. grid points
/
! io_nml: general switches for model I/O -------------------------------------
&io_nml
 itype_pres_msl              =                          4         ! method for computation of mean sea level pressure
 itype_rh                    =                          1         ! method for computation of relative humidity
 restart_write_mode          =     'joint procs multifile'
/
! output_nml: specifies an output stream --------------------------------------
&output_nml
 filetype                    =                          4 ! output format: 2=GRIB2, 4=NETCDFv2
 dom                         =                          1 ! write domain 1 only
 output_start                =  "2010-04-14T00:00:00Z"
 output_end                  =  "2010-04-18T00:00:00Z"
 output_interval             =  "PT1H"
 steps_per_file              =                          1 ! number of steps per file
 include_last                =                      .TRUE.
 output_filename             =             'art-eyja-4000'! change to 9000 for second simulation
 remap                       = 1                          ! 1: remap to lat-lon grid
 reg_lon_def                 = -180.,0.5,180.
 reg_lat_def                 = 90.,-0.5, -90.
 ml_varlist                  = 'group:ART_AEROSOL','rho','pres','temp', 'z_mc','z_ifc', 'u', 'v', 'w'
/
! art_nml: Aerosols and Reactive Trace gases extension
&art_nml
 lart_diag_out   = .TRUE.
 lart_chem       = .FALSE.
 lart_chemtracer = .FALSE.
 lart_aerosol    = .TRUE.
 lart_pntSrc     = .TRUE.
 cart_aerosol_xml    = '${XMLDIR}/Ex03_aerotracer.xml'
 cart_modes_xml      = '${XMLDIR}/Ex03_modes.xml'
 cart_pntSrc_xml     = '${XMLDIR}/Ex03_pntSrc-4000.xml'    ! change to 9000 for second simulation
/
EOF

## 4. Job Settings
Insert the correct project account.

In [ ]:
# ----------------------------------------------------------------------
# run the model!
# ----------------------------------------------------------------------

cat > ${OUTDIR}/job_ICON << ENDFILE
#!/bin/bash
#SBATCH --job-name=EXP03_ART          # Specify job name
#SBATCH --partition=compute            # Specify partition name
#SBATCH --nodes=8                    # Specify number of nodes
#SBATCH --ntasks-per-node=128          # Specify number of (MPI) tasks on each node
#SBATCH --time=01:00:00                 # Set a limit on the total run time
#SBATCH --account=????                # Charge resources on this project account

unset SLURM_EXPORT_ENV 
unset SLURM_MEM_PER_NODE
unset SBATCH_EXPORT

export LD_LIBRARY_PATH="/usr/lib:/usr/lib64:/sw/spack-levante/netcdf-c-4.8.1-2k3cmu/lib:/sw/spack-levante/netcdf-fortran-4.5.3-k6xq5g/lib:/sw/spack-levante/hdf5-1.12.1-tvymb5/lib:/sw/spack-levante/eccodes-2.21.0-3ehkbb/lib64:/sw/spack-levante/intel-oneapi-mkl-2022.0.1-ttdktf/mkl/2022.0.1/lib/intel64/"

export OMPI_MCA_pml="ucx"
export OMPI_MCA_btl=self
export OMPI_MCA_osc="pt2pt"
export UCX_IB_ADDR_TYPE=ib_global

export OMPI_MCA_coll="^ml,hcoll"
export OMPI_MCA_coll_hcoll_enable="0"
export HCOLL_ENABLE_MCAST_ALL="0"
export HCOLL_MAIN_IB=mlx5_0:1
export UCX_NET_DEVICES=mlx5_0:1
export UCX_TLS=mm,knem,cma,dc_mlx5,dc_x,self
export UCX_UNIFIED_MODE=y
export HDF5_USE_FILE_LOCKING=FALSE
export OMPI_MCA_io="romio321"
export UCX_HANDLE_ERRORS=bt

ulimit -s 102400
ulimit -c 0

srun -l --cpu_bind=cores --distribution=block:cyclic --propagate=STACK,CORE ./icon.exe

ENDFILE

## 5. Submit the job

In [ ]:
cd $OUTDIR && chmod +x job_ICON && sbatch job_ICON

## 6. Check the job status and output data

In [ ]:
squeue -u $USER

<div class="alert alert-success">
    <b style="color:#2d4b9b;">Check the slurm-file for errors</b>
    <ul>
      <li>Open an interactive Linux terminal by clicking the following button <button data-commandlinker-command="terminal:create-new">Create Terminal</button></li>
      <li>Go to your output directory using the cd command</li>
      <li>Open the slurm file with e.g., vim</li>
</div>